# Project Work XAI

In [1]:
import os
print(os.getpid())

849212


In [2]:
import os
print("Il mio PID:", os.getpid())
print("Il PID del processo Padre (chi mi ha avviato):", os.getppid())

Il mio PID: 849212
Il PID del processo Padre (chi mi ha avviato): 847978


In [3]:
!nvidia-smi

Wed May 13 20:02:14 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 2070 ...    Off | 00000000:04:00.0 Off |                  N/A |
| 39%   32C    P8              12W / 215W |     16MiB /  8192MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
import torch

# Check CUDA support (if Python sees the GPU)
cuda_available = torch.cuda.is_available()
print(f"CUDA support available: {cuda_available}")

if cuda_available:
    # how many GPUs
    number_gpu = torch.cuda.device_count()
    print(f"Number of GPU detected: {number_gpu}")

    # GPU name
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU model: {gpu_name}")
    
    # total memory
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9 # in GB
    print(f"Total VRAM memory: {total_memory:.2f} GB")
else:
    print("WARNING: The code is only using the CPU. Check your drivers or Torch installation.")

CUDA support available: True
Number of GPU detected: 2
GPU model: NVIDIA GeForce RTX 2080 SUPER
Total VRAM memory: 8.36 GB


In [5]:
device = torch.device("cuda")

x = torch.randn(1000, 1000).to(device)

print(x.device)

cuda:0


## Environment management

In [6]:
import os
import sys

# detecting environment
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

# setting filesystem and paths
if is_colab():
    print("Running in Google Colab detected.")
    # cloning repository
    repo_url = "https://github.com/Epot12/Lab_XAI.git"
    repo_name = "Lab_XAI"

    # cloning only if it hasn't already been done in this session
    if not os.path.exists(repo_name):
        !git clone {repo_url}

    # changing the working directory to the project directory
    os.chdir(repo_name)

    # Adding the folder to the system paths
    sys.path.append(os.getcwd())
    print(f"Working directory set to: {os.getcwd()}")

else:
    from pathlib import Path
    print("Local execution detected.")
    def project_root():
        current = Path.cwd().resolve()
        for path in [current] + list(current.parents):
            if (path / ".git").exists():
                return path
        raise RuntimeError("Project root not found")

    os.chdir(project_root())
    sys.path.append(str(project_root()))
    print(f"Working directory set to: {os.getcwd()}")

Local execution detected.
Working directory set to: /home/emiliano/projects/project_1/Lab_XAI


In [7]:
# installing uv

if is_colab():
    print("Installing uv...")
    !curl -LsSf https://astral.sh/uv/install.sh | sh

    # Reload the path to show the UV track

    os.environ['PATH'] += ':/root/.cargo/bin'

    print("Environment synchronization...")
    # Synchronize Colab's system environment with local environment dependencies
    !uv pip install --system -r pyproject.toml

In [9]:
from k_validation.K_fold import *
from models.neural_network import *
from utils.data_processing import *

## Data extraction
In order to assess the importance of Solar Flux variable for predictions, in this evaluation such variable will be kept in the dataset and model performance will be evaluated considering data obtained in this way.

In [12]:
X, y = load_and_preprocess_data(dataset_path="Data/MARSIS_historical_dataset.csv", orbit_path="Data/orbit_to_remove", keep_flux=True)

## Creating Data Splits and Training

In [13]:
k_fold_dict = k_fold_val(X, y, NeurNet, n_splits=10, epochs=1000, patience=10, exp_name="exp_1")


--- Starting FOLD 0 ---


TypeError: 'module' object is not callable